# Train the dog detector on Google Colab (free GPU)

1. **Runtime → Change runtime type → T4 GPU**
2. Set `REPO_URL` below to your GitHub repository, then **Runtime → Run all**.
3. At the end a `models.zip` downloads. Copy its contents into the `models/` folder on the Raspberry Pi.

Training on ~4,000 COCO dog photos + hard negatives takes roughly 1–2 hours for 60 epochs on a T4.

In [ ]:
REPO_URL = "https://github.com/mazenelnaghy-code/dog-detection"  # <-- change to your repository
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

## 1. Build the dataset
Downloads only the COCO images that contain dogs (+ people/cats as negatives).
Use `--max-dogs 1500` for a faster run.

**Optional — your own photos:** label them on [Roboflow](https://roboflow.com) (class name `dog`),
export as *YOLOv8*, upload the unzipped folder to Colab and add `--extra /content/my_dogs`.

In [ ]:
!python training/prepare_coco_dogs.py --out datasets/dogs

## 2. Train (transfer learning from the COCO-pretrained YOLO11n)

In [ ]:
!python training/train.py --data datasets/dogs/data.yaml --epochs 60 --batch 32 --device 0

## 3. Check a few predictions

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob
model = YOLO("models/dog_yolo11n.pt")
for path in sorted(glob.glob("datasets/dogs/images/val/*.jpg"))[:4]:
    model.predict(path, save=True, project="preview", name="val", exist_ok=True, verbose=False)
for path in sorted(glob.glob("preview/val/*.jpg"))[:4]:
    display(Image(filename=path, width=480))

## 4. Export for the Raspberry Pi (NCNN) and download

In [ ]:
!python training/export.py --weights models/dog_yolo11n.pt --format ncnn --imgsz 640 --out models
!zip -qr models.zip models
from google.colab import files
files.download("models.zip")

On the Pi, set in `config/pi.yaml` (or `config/local.yaml`):
```yaml
model:
  path: models/dog_yolo11n_ncnn_model
  imgsz: 640
```